[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_04/notebook_4_4_cross_validation.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 4.4: Cross-Validation Strategies for Healthcare ML

**Chapter 4: Data Preparation**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement standard k-fold cross-validation
2. Use stratified CV for class-imbalanced datasets
3. Apply temporal CV for time series data
4. Design grouped CV for patient-level splitting
5. Understand when each CV strategy is appropriate

## Clinical Context

**Cross-validation in healthcare has unique challenges**:
- **Class imbalance**: Rare outcomes (sepsis, mortality) require stratification
- **Temporal dependence**: Models must be validated on future data
- **Patient-level grouping**: Multiple records per patient require grouped splits
- **Site effects**: Multi-site data requires leave-one-site-out CV

**Wrong CV strategy leads to**:
- Overly optimistic performance estimates
- Models that fail in deployment
- Data leakage from train to test

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit,
    cross_val_score, cross_validate
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported")

## 1. Generate Test Datasets

We'll create three different scenarios:
1. Standard tabular data
2. Time series data
3. Grouped data (multiple records per patient)

In [ ]:
def generate_standard_data(n_patients=1000, prevalence=0.10):
    """
    Generate standard tabular healthcare data with class imbalance.
    """
    age = np.random.normal(65, 15, n_patients)
    age = np.clip(age, 18, 100)

    feature1 = np.random.normal(100, 20, n_patients)
    feature2 = np.random.normal(50, 10, n_patients)
    feature3 = np.random.choice([0, 1], n_patients, p=[0.7, 0.3])

    # Outcome with class imbalance
    risk_score = 0.05 * (age - 65) + 0.01 * (feature1 - 100) + 0.5 * feature3
    prob = 1 / (1 + np.exp(-risk_score))

    # Adjust to get target prevalence
    threshold = np.percentile(prob, (1 - prevalence) * 100)
    outcome = (prob > threshold).astype(int)

    df = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(n_patients)],
        'age': age,
        'feature1': feature1,
        'feature2': feature2,
        'feature3': feature3,
        'outcome': outcome
    })

    return df

def generate_time_series_data(n_patients=200, timesteps_per_patient=24):
    """
    Generate time series data (e.g., ICU vitals).
    """
    data = []

    for i in range(n_patients):
        # Patient baseline
        baseline_risk = np.random.rand()

        for t in range(timesteps_per_patient):
            # Vitals that deteriorate over time if at risk
            hr = 70 + baseline_risk * 30 + t * baseline_risk
            sbp = 120 - baseline_risk * 20 - t * baseline_risk * 0.5

            hr += np.random.normal(0, 5)
            sbp += np.random.normal(0, 10)

            data.append({
                'patient_id': f'P{i:04d}',
                'timestamp': t,
                'heart_rate': hr,
                'sbp': sbp,
                'baseline_risk': baseline_risk
            })

    df = pd.DataFrame(data)

    # Outcome at last timestamp
    df['outcome'] = (df['baseline_risk'] > 0.5).astype(int)

    return df

def generate_grouped_data(n_patients=500, visits_per_patient=(1, 5)):
    """
    Generate grouped data (multiple visits per patient).
    """
    data = []

    for i in range(n_patients):
        # Patient-level characteristics
        patient_risk = np.random.rand()
        n_visits = np.random.randint(visits_per_patient[0], visits_per_patient[1] + 1)

        for v in range(n_visits):
            # Visit-level measurements
            feature1 = 100 + patient_risk * 50 + np.random.normal(0, 10)
            feature2 = 50 + patient_risk * 20 + np.random.normal(0, 5)

            # Outcome more likely with higher patient risk
            outcome = (np.random.rand() < patient_risk * 0.3)

            data.append({
                'patient_id': f'P{i:04d}',
                'visit_id': v,
                'feature1': feature1,
                'feature2': feature2,
                'outcome': outcome
            })

    return pd.DataFrame(data)

# Generate all datasets
df_standard = generate_standard_data(1000, prevalence=0.10)
df_timeseries = generate_time_series_data(200, timesteps_per_patient=24)
df_grouped = generate_grouped_data(500, visits_per_patient=(1, 5))

print("Standard dataset:")
print(f"  Patients: {len(df_standard)}")
print(f"  Outcome prevalence: {df_standard['outcome'].mean():.1%}")

print("\nTime series dataset:")
print(f"  Total records: {len(df_timeseries)}")
print(f"  Unique patients: {df_timeseries['patient_id'].nunique()}")
print(f"  Outcome prevalence: {df_timeseries.groupby('patient_id')['outcome'].first().mean():.1%}")

print("\nGrouped dataset:")
print(f"  Total visits: {len(df_grouped)}")
print(f"  Unique patients: {df_grouped['patient_id'].nunique()}")
print(f"  Visits per patient: {df_grouped.groupby('patient_id').size().mean():.1f}")
print(f"  Outcome prevalence: {df_grouped['outcome'].mean():.1%}")

## 2. Standard K-Fold Cross-Validation

In [ ]:
# Prepare standard data
X = df_standard[['age', 'feature1', 'feature2', 'feature3']]
y = df_standard['outcome']

# Standard K-Fold (WARNING: May not preserve class balance in folds!)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

print("="*70)
print("STANDARD K-FOLD CROSS-VALIDATION")
print("="*70)

# Check class distribution in each fold
for fold, (train_idx, val_idx) in enumerate(kfold.split(X)):
    train_prevalence = y.iloc[train_idx].mean()
    val_prevalence = y.iloc[val_idx].mean()

    print(f"\nFold {fold + 1}:")
    print(f"  Train size: {len(train_idx)}, prevalence: {train_prevalence:.1%}")
    print(f"  Val size: {len(val_idx)}, prevalence: {val_prevalence:.1%}")
    print(f"  Difference: {abs(train_prevalence - val_prevalence):.1%}")

# Cross-validation scores
clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
cv_scores = cross_val_score(clf, X, y, cv=kfold, scoring='roc_auc')

print(f"\nAUROC scores per fold: {cv_scores}")
print(f"Mean AUROC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print("="*70)

## 3. Stratified K-Fold (For Class Imbalance)

In [ ]:
# Stratified K-Fold (RECOMMENDED for imbalanced data)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("="*70)
print("STRATIFIED K-FOLD CROSS-VALIDATION")
print("="*70)

# Check class distribution in each fold
for fold, (train_idx, val_idx) in enumerate(stratified_kfold.split(X, y)):
    train_prevalence = y.iloc[train_idx].mean()
    val_prevalence = y.iloc[val_idx].mean()

    print(f"\nFold {fold + 1}:")
    print(f"  Train size: {len(train_idx)}, prevalence: {train_prevalence:.1%}")
    print(f"  Val size: {len(val_idx)}, prevalence: {val_prevalence:.1%}")
    print(f"  Difference: {abs(train_prevalence - val_prevalence):.1%}")

# Cross-validation scores
cv_scores_stratified = cross_val_score(clf, X, y, cv=stratified_kfold, scoring='roc_auc')

print(f"\nAUROC scores per fold: {cv_scores_stratified}")
print(f"Mean AUROC: {cv_scores_stratified.mean():.3f} ± {cv_scores_stratified.std():.3f}")

print("\n💡 Stratified CV maintains class balance across folds!")
print("="*70)

## 4. Temporal Cross-Validation (For Time Series)

In [ ]:
# Prepare time series data (aggregate per patient)
df_ts_agg = df_timeseries.groupby('patient_id').agg({
    'heart_rate': ['mean', 'std', 'max'],
    'sbp': ['mean', 'std', 'min'],
    'outcome': 'first',
    'timestamp': 'max'  # Use as time index
}).reset_index()

df_ts_agg.columns = ['patient_id', 'hr_mean', 'hr_std', 'hr_max',
                     'sbp_mean', 'sbp_std', 'sbp_min', 'outcome', 'time_index']

# Sort by time
df_ts_agg = df_ts_agg.sort_values('time_index').reset_index(drop=True)

X_ts = df_ts_agg[['hr_mean', 'hr_std', 'hr_max', 'sbp_mean', 'sbp_std', 'sbp_min']]
y_ts = df_ts_agg['outcome']

# TimeSeriesSplit (respects temporal ordering)
tscv = TimeSeriesSplit(n_splits=5)

print("="*70)
print("TEMPORAL CROSS-VALIDATION (Time Series Split)")
print("="*70)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_ts)):
    train_time_range = (df_ts_agg.iloc[train_idx]['time_index'].min(),
                        df_ts_agg.iloc[train_idx]['time_index'].max())
    val_time_range = (df_ts_agg.iloc[val_idx]['time_index'].min(),
                      df_ts_agg.iloc[val_idx]['time_index'].max())

    print(f"\nFold {fold + 1}:")
    print(f"  Train: {len(train_idx)} patients, time range: {train_time_range}")
    print(f"  Val: {len(val_idx)} patients, time range: {val_time_range}")
    print(f"  ✓ Validation always AFTER training (no future leakage)")

# Cross-validation scores
cv_scores_temporal = cross_val_score(clf, X_ts, y_ts, cv=tscv, scoring='roc_auc')

print(f"\nAUROC scores per fold: {cv_scores_temporal}")
print(f"Mean AUROC: {cv_scores_temporal.mean():.3f} ± {cv_scores_temporal.std():.3f}")
print("\n💡 Temporal CV ensures no future data leaks into training!")
print("="*70)

## 5. Grouped Cross-Validation (Patient-Level Splitting)

In [ ]:
# Prepare grouped data
X_grouped = df_grouped[['feature1', 'feature2']]
y_grouped = df_grouped['outcome']
groups = df_grouped['patient_id']

# GroupKFold (ensures all visits from same patient in same fold)
group_kfold = GroupKFold(n_splits=5)

print("="*70)
print("GROUPED CROSS-VALIDATION (Patient-Level Splitting)")
print("="*70)

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_grouped, y_grouped, groups)):
    train_patients = groups.iloc[train_idx].unique()
    val_patients = groups.iloc[val_idx].unique()

    # Check for leakage
    overlap = set(train_patients) & set(val_patients)

    print(f"\nFold {fold + 1}:")
    print(f"  Train: {len(train_idx)} visits from {len(train_patients)} patients")
    print(f"  Val: {len(val_idx)} visits from {len(val_patients)} patients")
    print(f"  Patient overlap: {len(overlap)} (should be 0!)")
    print(f"  ✓ No patient appears in both train and val")

# Cross-validation scores
cv_scores_grouped = cross_val_score(clf, X_grouped, y_grouped,
                                    cv=group_kfold, groups=groups, scoring='roc_auc')

print(f"\nAUROC scores per fold: {cv_scores_grouped}")
print(f"Mean AUROC: {cv_scores_grouped.mean():.3f} ± {cv_scores_grouped.std():.3f}")
print("\n💡 Grouped CV prevents patient-level leakage!")
print("="*70)

## 6. Comparison: Impact of Wrong CV Strategy

In [ ]:
# Demonstrate impact of using wrong CV for grouped data
print("="*70)
print("DANGER: Using Standard K-Fold on Grouped Data")
print("="*70)

# WRONG: Standard K-Fold on grouped data (allows same patient in train and val)
kfold_wrong = KFold(n_splits=5, shuffle=True, random_state=42)

leakage_detected = []
for fold, (train_idx, val_idx) in enumerate(kfold_wrong.split(X_grouped)):
    train_patients = set(groups.iloc[train_idx].unique())
    val_patients = set(groups.iloc[val_idx].unique())
    overlap = train_patients & val_patients
    leakage_detected.append(len(overlap))

    if fold == 0:  # Show first fold detail
        print(f"\nFold {fold + 1} (WRONG approach):")
        print(f"  Train patients: {len(train_patients)}")
        print(f"  Val patients: {len(val_patients)}")
        print(f"  ❌ PATIENT OVERLAP: {len(overlap)} patients appear in BOTH train and val!")

cv_scores_wrong = cross_val_score(clf, X_grouped, y_grouped, cv=kfold_wrong, scoring='roc_auc')

print(f"\nWrong CV (standard K-Fold): AUROC = {cv_scores_wrong.mean():.3f} ± {cv_scores_wrong.std():.3f}")
print(f"Correct CV (GroupKFold): AUROC = {cv_scores_grouped.mean():.3f} ± {cv_scores_grouped.std():.3f}")
print(f"\n💥 Optimism bias: {(cv_scores_wrong.mean() - cv_scores_grouped.mean()):.3f}")
print(f"Average patient leakage per fold: {np.mean(leakage_detected):.0f} patients")
print("\n⚠️ Using wrong CV leads to overly optimistic performance estimates!")
print("="*70)

## 7. Visualize CV Strategies

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Helper function to plot fold splits
def plot_cv_splits(cv, X, y, groups=None, ax=None, title=''):
    n_splits = cv.get_n_splits(X, y, groups)

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups)):
        indices = np.zeros(len(X))
        indices[val_idx] = 1  # Validation = 1

        ax.scatter(range(len(X)), [fold] * len(X),
                  c=indices, cmap='RdYlGn', s=10, alpha=0.8)

    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Fold')
    ax.set_title(title, fontweight='bold')
    ax.set_yticks(range(n_splits))
    ax.set_yticklabels([f'Fold {i+1}' for i in range(n_splits)])
    ax.grid(True, alpha=0.3, axis='x')

# Standard K-Fold
plot_cv_splits(kfold, X, y, ax=axes[0, 0],
               title='Standard K-Fold\n(Red=Val, Green=Train)')

# Stratified K-Fold
plot_cv_splits(stratified_kfold, X, y, ax=axes[0, 1],
               title='Stratified K-Fold\n(Maintains class balance)')

# Temporal Split
plot_cv_splits(tscv, X_ts, y_ts, ax=axes[1, 0],
               title='Temporal Split\n(Training always before validation)')

# Grouped Split
plot_cv_splits(group_kfold, X_grouped, y_grouped, groups=groups, ax=axes[1, 1],
               title='Grouped K-Fold\n(Patients never split across folds)')

plt.tight_layout()
plt.show()

## 8. Decision Tree: Which CV Strategy to Use?

In [ ]:
print("="*70)
print("DECISION TREE: CHOOSING THE RIGHT CV STRATEGY")
print("="*70)

print("\n1. DO YOU HAVE TIME SERIES DATA?")
print("   YES → Use TimeSeriesSplit or forward chaining")
print("   NO → Continue to Q2")

print("\n2. DO YOU HAVE GROUPED DATA (multiple records per patient/site)?")
print("   YES → Use GroupKFold")
print("   NO → Continue to Q3")

print("\n3. DO YOU HAVE CLASS IMBALANCE (rare outcome)?")
print("   YES → Use StratifiedKFold")
print("   NO → Standard KFold is acceptable")

print("\n" + "="*70)
print("SPECIAL CASES")
print("="*70)

print("\n4. MULTI-SITE DATA")
print("   → Use Leave-One-Group-Out (site as group)")
print("   → Tests generalization to new sites")

print("\n5. EXTREMELY RARE OUTCOMES (<1%)")
print("   → Use StratifiedKFold with more folds (k=10)")
print("   → Or stratified train/val/test split")

print("\n6. COMBINED CHALLENGES (time series + grouped)")
print("   → Custom CV: GroupKFold + temporal ordering")
print("   → Ensure groups respect time boundaries")

print("\n" + "="*70)

## 9. Common Pitfalls and How to Avoid Them

In [ ]:
print("="*70)
print("COMMON CROSS-VALIDATION PITFALLS")
print("="*70)

print("\n❌ PITFALL 1: Preprocessing before splitting")
print("   WRONG: Scale entire dataset → split → evaluate")
print("   RIGHT: Split → scale training set → apply to validation")
print("   Impact: Leakage of validation statistics into training")

print("\n❌ PITFALL 2: Using K-Fold on time series")
print("   WRONG: Random folds on temporal data")
print("   RIGHT: TimeSeriesSplit (always validate on future)")
print("   Impact: Model sees future data during training (impossible in practice)")

print("\n❌ PITFALL 3: Ignoring patient grouping")
print("   WRONG: Standard K-Fold on multiple visits per patient")
print("   RIGHT: GroupKFold with patient_id as group")
print("   Impact: Same patient in train and val → overfitting to individual patients")

print("\n❌ PITFALL 4: Not stratifying imbalanced data")
print("   WRONG: K-Fold on 1% prevalence outcome")
print("   RIGHT: StratifiedKFold")
print("   Impact: Some folds may have 0% or 3% prevalence → unstable metrics")

print("\n❌ PITFALL 5: Feature selection within CV loop")
print("   WRONG: Select features on full dataset → CV")
print("   RIGHT: Feature selection within each CV fold")
print("   Impact: Validation set influences feature selection → optimism bias")

print("\n❌ PITFALL 6: Hyperparameter tuning on test set")
print("   WRONG: Tune hyperparameters using test set")
print("   RIGHT: Use nested CV or separate val set for tuning")
print("   Impact: Test set no longer independent → overfitted hyperparameters")

print("\n" + "="*70)

## 10. Key Takeaways

### What We Learned

1. **Standard K-Fold**:
   - Randomly splits data into k folds
   - Simple but may not preserve class balance
   - Use when: Large balanced dataset, no special structure

2. **Stratified K-Fold**:
   - Maintains class distribution in each fold
   - CRITICAL for imbalanced healthcare outcomes
   - Use when: Rare outcomes (sepsis, mortality, etc.)

3. **Temporal CV**:
   - Respects time ordering
   - Training always before validation
   - Use when: Time series, ICU data, longitudinal studies

4. **Grouped CV**:
   - Keeps related records together
   - Prevents patient-level leakage
   - Use when: Multiple visits/records per patient

5. **Impact of Wrong CV**:
   - Optimism bias: 0.02-0.10 AUROC inflation
   - Models fail in deployment
   - Cannot trust performance estimates

### Clinical Considerations

- **Deployment reality**: CV should mimic how model will be used
- **Temporal validation**: Always validate on future patients
- **Site generalization**: Test on held-out hospitals
- **Patient privacy**: Never split same patient across train/test
- **Rare outcomes**: Ensure sufficient cases in each fold

### Best Practices

1. **Choose CV strategy based on data structure**, not convenience
2. **Use stratification for imbalanced outcomes** (<20% prevalence)
3. **Respect temporal ordering** for time series data
4. **Group by patient** when multiple records per patient
5. **Nested CV** for hyperparameter tuning (inner loop) + evaluation (outer loop)
6. **Preprocessing inside CV folds**, never before splitting
7. **Report CV strategy** in publications for reproducibility

### Connection to Journey Stories

**Marcus (Sepsis)**: Temporal CV critical - model must predict future sepsis from current/past vitals only (see Notebook 7.3)

**Jamal (Lung Nodules)**: Grouped CV by patient - multiple CT slices per patient require patient-level splitting (see Notebook 6.5)

**All Journeys**: Stratified CV for rare outcomes ensures representative performance estimates

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 4: Data Preparation)*

## Exercises

1. **Nested CV**: Implement nested cross-validation where the outer loop evaluates performance and the inner loop tunes hyperparameters.

2. **Leave-One-Site-Out**: Simulate multi-site data and implement leave-one-site-out CV to test generalization to new hospitals.

3. **Combined Temporal + Grouped**: Create dataset with time series AND multiple visits per patient. Design appropriate CV strategy.

4. **Leakage Detection**: Deliberately introduce preprocessing before splitting. Quantify the optimism bias created.

5. **Bootstrapping**: Implement bootstrap resampling as an alternative to CV. Compare confidence intervals from bootstrap vs k-fold.